In [ ]:
!pip install transformers torch -q

from transformers import pipeline
from IPython.display import HTML, display
import matplotlib.pyplot as plt

In [ ]:
classifier = pipeline('sentiment-analysis')

zdania = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
    "The food was great but the service was terrible.",
    "Not bad at all!",
    "Oh great, another Monday.",
    "This is sick!",
]

wyniki = classifier(zdania)

for zdanie, wynik in zip(zdania, wyniki):
    print(f"{zdanie[:50]:<52} {wynik['label']:<10} {wynik['score']:.1%}")

In [ ]:
scores = [w['score'] if w['label'] == 'POSITIVE' else -w['score'] for w in wyniki]
kolory = ['#2ecc71' if s > 0 else '#e74c3c' for s in scores]
etykiety = [z[:40] for z in zdania]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(etykiety, scores, color=kolory)
ax.axvline(0, color='gray', linewidth=1)
ax.set_xlim(-1.05, 1.05)
ax.set_xlabel('← NEGATIVE          POSITIVE →')
ax.set_title('Sentyment i pewność modelu')
plt.tight_layout()
plt.show()

In [ ]:
ner = pipeline('ner', aggregation_strategy='simple')

TEKST_NER = """
Marie Curie was born in Warsaw, Poland in 1867. She studied at the University
of Paris and later worked at the Sorbonne. In 1903 she received the Nobel Prize
in Physics together with her husband Pierre Curie and Henri Becquerel.
In 1911 she won a second Nobel Prize, this time in Chemistry.
"""

wyniki_ner = ner(TEKST_NER)

for e in wyniki_ner:
    print(f"{e['word']:<25} {e['entity_group']:<6} {e['score']:.1%}")

In [ ]:
kolory_ner = {'PER': '#AED6F1', 'ORG': '#A9DFBF', 'LOC': '#F9E79F', 'MISC': '#F1948A'}

tekst_html = TEKST_NER.strip()
for e in sorted(wyniki_ner, key=lambda x: x['start'], reverse=True):
    kolor = kolory_ner.get(e['entity_group'], '#ddd')
    tag = f'<mark style="background:{kolor};padding:2px 6px;border-radius:4px">{e["word"]}</mark>'
    tekst_html = tekst_html[:e['start']] + tag + tekst_html[e['end']:]

display(HTML(f'<div style="font-size:16px;line-height:2.2">{tekst_html}</div>'))

In [ ]:
qa = pipeline('question-answering')

KONTEKST = """
The transformer architecture was introduced in 2017 in the paper
'Attention Is All You Need' by Vaswani et al. at Google Brain.
Unlike recurrent neural networks, transformers process all tokens
simultaneously using self-attention mechanisms, which allows them
to be trained much faster on modern GPU hardware.
BERT was introduced by Google in 2018 and became the dominant
model for language understanding tasks. GPT was developed by
OpenAI, also in 2018, and focuses on text generation.
The key difference is that BERT uses a bidirectional encoder,
while GPT uses a unidirectional decoder.
"""

pytania = [
    "When was the transformer introduced?",
    "Who created BERT?",
    "What is the key difference between BERT and GPT?",
    "Why are transformers faster than RNNs?",
    "What is the price of GPT-4?",
]

for p in pytania:
    odp = qa(question=p, context=KONTEKST)
    print(f"Q: {p}")
    print(f"A: {odp['answer']}  ({odp['score']:.1%})\n")

In [ ]:
zero_shot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

NAGLOWKI = [
    "Government announces new austerity measures amid rising debt.",
    "Scientists discover potential cure for Alzheimer's disease.",
    "Stock markets plunge following central bank interest rate decision.",
    "New study links social media use to increased anxiety in teenagers.",
    "Climate summit ends without binding agreement on emissions.",
    "Police used tear gas to disperse the demonstrators.",
    "The prime minister called for dialogue and calm.",
]

KATEGORIE = ['politics', 'science', 'economy', 'society', 'environment']

for naglowek in NAGLOWKI:
    wynik = zero_shot(naglowek, candidate_labels=KATEGORIE)
    print(f"{naglowek[:55]:<57} {wynik['labels'][0]:<12} {wynik['scores'][0]:.1%}")

In [ ]:
TEKSTY = [
    "Thousands flee as violence erupts at the border.",
    "New policy offers path to citizenship for skilled migrants.",
    "Local communities welcome refugees with open arms.",
    "Government struggles to control illegal border crossings.",
    "Asylum seekers bring cultural diversity to host countries.",
    "Migrant workers fill critical gaps in the labor market.",
]

FRAMING = ['humanitarian', 'security threat', 'economic', 'cultural', 'legal/policy']

for tekst in TEKSTY:
    wynik = zero_shot(tekst, candidate_labels=FRAMING)
    top = wynik['labels'][0]
    print(f"{tekst:<55} -> {top} ({wynik['scores'][0]:.0%})")

In [ ]:
generator = pipeline('text-generation', model='gpt2')

PROMPT = 'The history of artificial intelligence began'

for temp in [0.3, 0.9, 1.5]:
    wynik = generator(PROMPT, max_new_tokens=40, temperature=temp,
                      do_sample=True, pad_token_id=50256)
    print(f"temp={temp}: {wynik[0]['generated_text']}\n")